# 🧾 burst_scan — free options data: collect, store, compare, analyse

Collects every **free** source of options data for the 5 study stocks, keeps it on disk, checks the sources against
each other, and — once enough history is stored — asks whether options data predicts the stocks' **direction**
(the question 1-minute bars could not answer, `DIRECTION_REPORT.md`).

| source | kind | what it gives | history |
|---|---|---|---|
| `cboe` | live, ~15 min delayed | whole chain: bid/ask, last, volume, open interest, IV, Greeks | from the day you start collecting |
| `yfinance` | live (Yahoo) | bid/ask, last, volume, open interest, IV (no Greeks); **placeholders outside US market hours** | from the day you start |
| `nasdaq` | live | bid/ask, last, volume, open interest (no IV) | from the day you start |
| `occ_oi` | live (OCC Series Search) | open interest per contract, as of the previous close | from the day you start |
| `dolthub_chain` | history (community DoltHub repo) | end-of-day bid/ask, IV, Greeks for ~100-200 near-the-money contracts per stock; no volume/OI; some days missing | 2019 → yesterday |
| `dolthub_vol` | history | daily IV and historical-vol levels per stock | 2019 → yesterday |
| `occ_volume` | history (OCC Volume Query) | daily volume per stock × customer / firm / market maker × put/call × exchange | **rolling 2 years only** — keep collecting |
| `marketdata_chain` | history (MarketData.app, free account; token `MARKETDATA_TOKEN` in the git-ignored `.env`) | full end-of-day chain, every expiry: bid/ask (+sizes), last, **volume and open interest per contract**, underlying price; IV/Greeks empty in history | **last 1 year only** on free / trial accounts — keep collecting |

Run it **top to bottom**; every step is incremental (stored days are never fetched again). Run §1 once a day,
ideally ~16:30 New York (after the close, before the next open) so every live source shows the same end of day —
schedule it, or the history of live sources will have holes. Code: `options_free.py` (tested in
`tests/test_options_free.py`). Data: `options_data/` (git-ignored; personal research use only — the free endpoints'
terms do not allow redistribution; DoltHub data is CC BY-SA 4.0).

In [ ]:
# If this import fails: select the "trading-simple" Jupyter kernel
# (see README "Quick start"), or run  uv sync --extra notebook
import importlib.util
import warnings
from datetime import date, timedelta

warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from trading_simple import plots, strategies
from trading_simple.config import load_settings
from trading_simple.data import load_panel
from trading_simple.db import make_engine

_spec = importlib.util.spec_from_file_location(
    "options_free", strategies.STRATEGIES_DIR / "burst_scan" / "options_free.py"
)
of = importlib.util.module_from_spec(_spec)
_spec.loader.exec_module(of)


def below(fig, y=-0.15):
    """Legend under the plot."""
    fig.update_layout(
        legend={"orientation": "h", "x": 0, "xanchor": "left", "y": y, "yanchor": "top"}
    )
    return fig


# --- 0 · the knobs --------------------------------------------------------------------------------
SYMBOLS = ["MU", "COIN", "CSCO", "AAL", "ANET"]
COLLECT_LIVE = True  # §1: take a snapshot of every live source now
BACKFILL = True  # §2: fetch missing history days (incremental; the first run takes ~1-2 h)
DOLT_START = "2023-01-01"  # DoltHub history from here (it goes back to 2019)
OCC_START = (date.today() - timedelta(days=729)).isoformat()  # OCC keeps only 2 years
MD_START = (date.today() - timedelta(days=364)).isoformat()  # MarketData.app: 1 year on free plans
MAX_DAYS_PER_RUN = None  # cap the days fetched per source and run (None = all missing days)
ANALYSIS_START, ANALYSIS_END = (
    "2023-03-01",
    "2026-04-30",
)  # §6; after Apr 2026 = the holdout, unread
MIN_DAYS = 250  # §6 runs only when the IV history has at least this many days in the window

## 1 · Live snapshot from every free source

One pull per source and stock (~2 minutes). A source that fails is reported and skipped; the others are still
stored. **Outside US market hours Yahoo serves placeholders** (bid = ask = 0, open interest 0, IV 0.00001 / 0.5) —
`options_free.clean_yahoo` marks them missing, so a pre-open run shows yfinance with no bid/ask/IV.

In [ ]:
if COLLECT_LIVE:
    snap_log = of.collect_snapshot(SYMBOLS)
    display(snap_log.pivot(index="underlying", columns="source", values="contracts"))
    errors = snap_log[snap_log.error != ""]
    print("errors:" if len(errors) else "no errors")
    if len(errors):
        display(errors)

## 2 · Backfill the history (incremental)

Every weekday not yet on disk. Holidays are stored as empty files so they are not asked again (except the last 4
days, which may simply not be published yet). The first full run takes ~1-2 hours (DoltHub ~1 s per day and table,
OCC ~3 s per day); later runs fetch only the new days.

In [ ]:
if BACKFILL:
    jobs = [("dolthub_vol", DOLT_START), ("dolthub_chain", DOLT_START), ("occ_volume", OCC_START)]
    if of.marketdata_token():
        jobs.append(("marketdata_chain", MD_START))
    else:
        print("marketdata_chain skipped: put MARKETDATA_TOKEN=... in the repo's .env")
    for src, start in jobs:
        log = of.backfill(src, start, symbols=SYMBOLS, max_days=MAX_DAYS_PER_RUN)
        bad = (log["error"] != "").sum() if len(log) else 0
        print(f"{src:14s} fetched {len(log):4d} days, {bad} errors")
        if bad:
            display(log[log.error != ""].head())

## 3 · What is stored, and is it complete?

`days_with_data` against the weekdays in the source's window: the gap is holidays (~10 a year) plus days the
source never published. DoltHub skips some trading days; the table lists the missing ones per month.

In [ ]:
inv = of.inventory()
display(inv)
dolt = of.load_history("dolthub_chain")
dvol = of.load_history("dolthub_vol")
occv = of.load_history("occ_volume")
mdc = of.load_history("marketdata_chain")
if len(dolt):
    have = set(dolt["date"].unique())
    wk = pd.bdate_range(DOLT_START, date.today() - timedelta(days=4)).strftime("%Y-%m-%d")
    missing = pd.Series([d for d in wk if d not in have], dtype=str)
    print(
        f"DoltHub chain: {len(have)} days with data, {len(missing)} weekdays without "
        f"(holidays included); per month:"
    )
    display(missing.str[:7].value_counts().sort_index().rename("weekdays missing").to_frame().T)
    per_day = dolt.groupby(["date", "underlying"]).size().unstack()
    fig = go.Figure(
        [
            go.Scatter(
                x=per_day.index, y=per_day[s], name=s, mode="lines", line={"color": plots.SERIES[i]}
            )
            for i, s in enumerate(per_day)
        ]
    )
    plots._base(
        fig, title="Fig 1 · DoltHub contracts per stock and day (the chain subset it stores)"
    )
    below(fig).show()

## 4 · The sources against each other (latest snapshot)

The same contract (root, expiry, strike, call/put) in two sources. `agree_share` = |difference| ≤ 1% of the value
(at least 0.01): prices to the cent, volume and open interest exactly, IV to 0.01. Read with the timing in mind:
Cboe and Nasdaq are both ~15 minutes delayed; OCC open interest is the **previous close's**, Cboe/Nasdaq/Yahoo
open interest can be updated intraday; Yahoo is only valid during market hours.

In [ ]:
stamps = of.snapshot_stamps()
print("snapshots stored:", len(stamps), "| latest:", stamps[-1] if stamps else None)
frames = of.load_snapshot()
if frames:
    display(of.coverage(frames).round(2))
    cmp = of.compare(frames)
    display(cmp.round(3))
    if len(cmp):
        pv = cmp.assign(pair=cmp.source_a + " vs " + cmp.source_b).pivot(
            index="pair", columns="field", values="agree_share"
        )
        fig = go.Figure(
            go.Heatmap(
                z=pv.values,
                x=pv.columns,
                y=pv.index,
                zmin=0,
                zmax=1,
                colorscale=plots.DIVERGING,
                text=pv.round(2).values,
                texttemplate="%{text}",
            )
        )
        plots._base(
            fig, title="Fig 2 · share of matched contracts on which two sources agree", height=360
        ).show()

**DoltHub vs the live sources.** DoltHub is an end-of-day snapshot. It is comparable with a live snapshot taken after
that day's close and before the next open (Cboe IV comes from Cboe's own model, DoltHub's from the uploader's — and
DoltHub stores one IV per strike for the call and the put, so expect level differences; the rank correlation is the
fair check).

In [ ]:
def after_close(stamp, day):
    t = pd.Timestamp(
        stamp.replace("_", " ")[:10] + " " + stamp[11:13] + ":" + stamp[13:15], tz="UTC"
    )
    d = pd.Timestamp(day, tz="UTC")
    return d + pd.Timedelta(hours=20, minutes=15) <= t <= d + pd.Timedelta(days=3, hours=13)


pairs = (
    [(s, d) for s in stamps for d in sorted(dolt["date"].unique())[-5:] if after_close(s, d)]
    if len(dolt)
    else []
)
if pairs:
    s, d = pairs[-1]
    live = of.load_snapshot(s)
    eod = {"dolthub_chain": dolt[dolt["date"] == d]}
    eod.update({k: v for k, v in live.items() if k in ("cboe", "nasdaq", "yfinance")})
    print(f"DoltHub end of day {d} vs the live snapshot {s}")
    display(of.compare(eod, fields=["bid", "ask", "iv", "delta"]).round(3))
else:
    print(
        "No live snapshot taken between a stored DoltHub day's close and the next open yet - "
        "run §1 after 16:30 New York and re-run this cell once DoltHub has that day."
    )

**OCC volume vs the chain sources.** A snapshot taken after the close carries the day's volume per contract; summed
per stock it should equal the OCC's cleared volume of that day (OCC publishes it the next morning).

In [ ]:
rows = []
for s in stamps:
    day = s[:10]
    live = of.load_snapshot(s)
    o = occv[occv["date"] == day] if len(occv) else pd.DataFrame()
    if o.empty:
        continue
    occ_tot = o.groupby("underlying")["quantity"].sum()
    for src in ("cboe", "nasdaq", "yfinance"):
        if src in live:
            v = live[src].groupby("underlying")["volume"].sum()
            for sym in occ_tot.index.intersection(v.index):
                rows.append(
                    {
                        "snapshot": s,
                        "source": src,
                        "underlying": sym,
                        "chain_volume": v[sym],
                        "occ_volume": occ_tot[sym],
                        "ratio": v[sym] / occ_tot[sym],
                    }
                )
display(
    pd.DataFrame(rows).round(3)
    if rows
    else "No snapshot day has OCC volume yet (OCC publishes the next morning; re-run tomorrow)."
)

**MarketData.app history against the other histories.** Same trading day, same contract: its end-of-day bid/ask
against DoltHub's, and its per-contract volume summed per stock against the OCC's cleared daily volume (the
clearing house is the reference; a chain that misses contracts sums low).

In [ ]:
if len(mdc):
    days = sorted(set(mdc["date"]) & set(dolt["date"]))[-20:] if len(dolt) else []
    if days:
        cmp_h = pd.concat(
            [
                of.compare(
                    {
                        "marketdata_chain": mdc[mdc["date"] == d],
                        "dolthub_chain": dolt[dolt["date"] == d],
                    },
                    fields=["bid", "ask"],
                ).assign(date=d)
                for d in days
            ]
        )
        display(cmp_h.groupby("field")[["matched", "agree_share", "rank_corr"]].mean().round(3))
    if len(occv):
        mv = mdc.groupby(["date", "underlying"])["volume"].sum().rename("chain_volume")
        ov = occv.groupby(["date", "underlying"])["quantity"].sum().rename("occ_volume")
        vv = pd.concat([mv, ov], axis=1, join="inner")
        vv["ratio"] = vv["chain_volume"] / vv["occ_volume"]
        print(f"{len(vv)} stock-days: chain volume / OCC volume, median per stock")
        display(
            vv.groupby("underlying")["ratio"].describe()[["count", "50%", "min", "max"]].round(3)
        )
else:
    print("No MarketData.app history stored (token missing or §2 not run).")

## 5 · Data-quality notes found while building this

* **Yahoo off-hours placeholders** (see §1): bid/ask/IV/open interest are zeros or constants outside US trading
  hours; only `last` and `volume` stay real. Collect during or right after the session.
* **Cboe and Nasdaq are the same feed**: they agree on 99-100% of contracts for every field they share — a second
  copy, not a second opinion.
* **Open interest** differs between OCC and the quote vendors on ~30% of contracts (rank correlation 0.99): timing
  of the daily update, not errors — OCC's is the clearing house's own figure, use it as the reference.
* **DoltHub** stores only a near-the-money subset (3-4 expiries, ~1 week to 2 months), one IV per strike for call and
  put, and misses some trading days: good for an IV / skew series, not for volume, flow or dealer gamma.
* **OCC volume** has no strikes, but it splits **customer / firm / market maker** — the free proxy for who traded.

## 6 · Analysis: does options data predict the next day's or week's direction?

Runs only when the IV history covers at least `MIN_DAYS` days in `ANALYSIS_START`..`ANALYSIS_END` (default: up to
2026-04-30 — the months after it are the study's untouched holdout). Features known at day *d*'s close; returns from
day *d+1*'s **open** to its close (`ret_1d`) or to the close 5 sessions later (`ret_5d`, scored on every 5th day so
the windows do not overlap):

* from DoltHub: `atm_iv` (30-day at-the-money IV), `skew` (25-delta put IV − 25-delta call IV), `term` (short minus
  30-day ATM IV), `d_atm_iv` / `d_skew` (one-day changes), `iv_hv` (IV − 20-day historical vol, `dolthub_vol`);
* from OCC: `cust_pc` (log customer put/call volume), `cust_share`, `vol_surge` (volume vs its previous 20 days).

Each feature is z-scored against its own stock's previous 60 days (so a stock's level is not a signal). Rank IC
pooled over the 5 stocks with a t-statistic **clustered by day**; per-stock ICs alongside. Published results predict:
steeper skew → lower returns, more put buying → lower returns, IV above HV → ambiguous.

In [ ]:
iv = (
    of.iv_features(dolt[(dolt["date"] >= ANALYSIS_START) & (dolt["date"] <= ANALYSIS_END)])
    if len(dolt)
    else pd.DataFrame()
)
ready = len(iv) and iv["date"].nunique() >= MIN_DAYS
print(
    f"IV history in the window: {iv['date'].nunique() if len(iv) else 0} days (need {MIN_DAYS}) ->",
    "running §6" if ready else "NOT complete yet - finish §2 first",
)

In [ ]:
if ready:
    iv = iv.sort_values(["underlying", "date"])
    iv["d_atm_iv"] = iv.groupby("underlying")["atm_iv"].diff()
    iv["d_skew"] = iv.groupby("underlying")["skew"].diff()
    feat = iv
    if len(dvol):
        hv = dvol.rename(columns={"act_symbol": "underlying"})[
            ["date", "underlying", "iv_current", "hv_current"]
        ]
        hv["iv_hv"] = hv["iv_current"].astype(float) - hv["hv_current"].astype(float)
        feat = feat.merge(
            hv[["date", "underlying", "iv_hv"]], on=["date", "underlying"], how="left"
        )
    if len(occv):
        feat = feat.merge(of.occ_features(occv), on=["date", "underlying"], how="left")
    FEATURES = [
        c
        for c in (
            "atm_iv",
            "skew",
            "term",
            "d_atm_iv",
            "d_skew",
            "iv_hv",
            "cust_pc",
            "cust_share",
            "vol_surge",
        )
        if c in feat
    ]
    feat = of.zscore_by_name(feat, FEATURES)
    daily = load_panel(
        make_engine(load_settings()),
        SYMBOLS,
        "1d",
        ANALYSIS_START,
        (pd.Timestamp(ANALYSIS_END) + pd.Timedelta(days=10)).date(),
    )
    fwd = of.forward_returns(daily, horizons=(1, 5))
    T = feat.merge(fwd, on=["date", "underlying"], how="inner")
    T = T[T["date"] <= ANALYSIS_END]
    Z = [f + "_z" for f in FEATURES]
    ic1 = of.ic_table(T, Z, ["ret_1d"])
    days5 = sorted(T["date"].unique())[::5]
    ic5 = of.ic_table(T[T["date"].isin(days5)], Z, ["ret_5d"])
    table = pd.concat([ic1, ic5], ignore_index=True).round(3)
    print(
        f"{T['date'].nunique()} days x {T['underlying'].nunique()} stocks; "
        f"{len(table)} feature x horizon cells: |t| >= 3 by chance "
        f"~{len(table) * 0.0027:.2f} times, |t| >= 2 ~{len(table) * 0.046:.1f}"
    )
    display(table)

In [ ]:
if ready:
    fig = make_subplots(
        rows=2,
        cols=1,
        shared_xaxes=True,
        vertical_spacing=0.06,
        subplot_titles=("30-day at-the-money IV", "skew: 25-delta put IV - 25-delta call IV"),
    )
    for i, (s, g) in enumerate(iv.groupby("underlying")):
        col = plots.SERIES[i % len(plots.SERIES)]
        fig.add_trace(go.Scatter(x=g["date"], y=g["atm_iv"], name=s, line={"color": col}), 1, 1)
        fig.add_trace(
            go.Scatter(x=g["date"], y=g["skew"], name=s, line={"color": col}, showlegend=False),
            2,
            1,
        )
    plots._base(fig, title="Fig 3 · the IV features per stock (DoltHub end of day)", height=620)
    below(fig, -0.08).show()
    fig = go.Figure()
    for i, (tgt, g) in enumerate(table.groupby("target")):
        fig.add_trace(go.Bar(x=g["feature"], y=g["t"], name=tgt, marker_color=plots.SERIES[i]))
    fig.add_hline(y=3, line_dash="dot")
    fig.add_hline(y=-3, line_dash="dot")
    plots._base(fig, title="Fig 4 · day-clustered t of each feature's IC (dotted: +-3)", height=380)
    below(fig).show()

### 6b · Open interest, volume and dealer gamma (MarketData.app history, ~1 year)

Per stock and day: `oi_pc` / `vol_pc` (log put/call open interest and volume), `d_oi` (open-interest change), `gex`
(dealer gamma imbalance in [-1, 1] under "dealers long calls, short puts"; gamma from the day's ATM IV), and
`gex_x_ret` = −`gex` × the sign of the day's own return: if negative dealer gamma amplifies moves, this predicts
**continuation** (positive IC) — the regime switch the price-only adaptive rules could not see. Only ~1 year of
history, so read the t-statistics as a first look, not a verdict; the holdout window is excluded as above.

In [ ]:
if len(mdc):
    ivd = of.iv_features(dolt) if len(dolt) else None
    mdf = of.md_features(mdc, ivd)
    daily_md = load_panel(
        make_engine(load_settings()),
        SYMBOLS,
        "1d",
        mdc["date"].min(),
        (pd.Timestamp(ANALYSIS_END) + pd.Timedelta(days=10)).date(),
    )
    ret0 = np.log(daily_md.close / daily_md.open_).stack().rename("ret0").reset_index()
    ret0.columns = ["date", "underlying", "ret0"]
    ret0["date"] = pd.to_datetime(ret0["date"]).dt.tz_localize(None).dt.strftime("%Y-%m-%d")
    M = mdf.merge(ret0, on=["date", "underlying"], how="left")
    M["gex_x_ret"] = -M["gex"] * np.sign(M["ret0"])
    MF = ["oi_pc", "vol_pc", "d_oi", "gex", "gex_x_ret"]
    M = of.zscore_by_name(M, MF, window=40)
    TM = M.merge(of.forward_returns(daily_md, horizons=(1, 5)), on=["date", "underlying"])
    TM = TM[TM["date"] <= ANALYSIS_END]
    print(
        f"MarketData features: {TM['date'].nunique()} days in the analysis window "
        f"({TM['date'].min()} .. {TM['date'].max()})"
    )
    ZM = [f + "_z" for f in MF]
    d5 = sorted(TM["date"].unique())[::5]
    table_md = pd.concat(
        [of.ic_table(TM, ZM, ["ret_1d"]), of.ic_table(TM[TM["date"].isin(d5)], ZM, ["ret_5d"])],
        ignore_index=True,
    ).round(3)
    display(table_md)

## 7 · Checks

In [ ]:
checks = [
    (
        "live sources responded",
        bool(COLLECT_LIVE) and len(snap_log) and (snap_log.error == "").mean() >= 0.75
        if COLLECT_LIVE
        else None,
    ),
    ("DoltHub history stored", len(dolt) > 0),
    ("OCC volume stored", len(occv) > 0),
    (
        "Cboe and Nasdaq agree (>= 95% of bids)",
        bool(
            len(frames)
            and len(cmp)
            and cmp[(cmp.source_a == "cboe") & (cmp.source_b == "nasdaq") & (cmp.field == "bid")]
            .agree_share.ge(0.95)
            .all()
        ),
    ),
    ("analysis window complete", bool(ready)),
]
if ready:
    best = table.loc[table["t"].abs().idxmax()]
    checks.append(
        (f"strongest IC: {best.feature} -> {best.target} (t {best.t:+.2f})", abs(best.t) >= 3)
    )
display(pd.DataFrame(checks, columns=["check", "ok"]))